# 102 - Pupils, Apertures, and the PSF

Notebooks to date have all used circle from the geometry package.  prysm
supports many other shapes, including central obscurations, polygonal
apertures, spiders, etc.  This notebook provides examples of several apertures
and their associated PSFs, and checks the clear-aperture case against the
analytic Airy formula prysm ships in `psf`.

We'll use the same 10mm, F/10 system at the HeNe wavelength as before.  We
begin as usual with imports and setting up the grid, as well as a circular
aperture

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from prysm.coordinates import make_xy_grid, cart_to_polar
from prysm.geometry import circle, annulus, spider, regular_polygon, antialias, annulus_sdf, multisample
from prysm.propagation import Wavefront
from prysm.wavelengths import HeNe
from prysm import psf as psfmod

# a 10 mm diameter, f/10 lens (100 mm efl) at the HeNe wavelength
EPD = 10.0          # entrance pupil diameter, mm
EFL = 100.0         # focal length, mm
FNO = EFL / EPD     # f/10
WVL = HeNe          # 0.6328 um

xi, eta = make_xy_grid(256, diameter=EPD)
r, t = cart_to_polar(xi, eta)
dx = xi[0, 1] - xi[0, 0]        # pupil sample spacing, mm
A = circle(EPD / 2, r)          # the clear circular aperture
airy_radius = 1.22 * WVL * FNO  # first dark ring, microns

## The clear aperture and the analytic Airy disk

`psf.airydisk` is the closed-form diffraction pattern of a clear circular
aperture, $\left[2 J_1(x)/x\right]^2$.  It should match the PSF we propagate from
the disk.  We overlay a slice through each:

In [ ]:
E = Wavefront.from_amp_and_phase(A, None, WVL, dx).focus(EFL, Q=4)
psf_prop = E.intensity
psf_prop.data = psf_prop.data / psf_prop.data.max()   # normalize peak to 1

# 1D slice through the PSF, and the analytic Airy disk on the same coordinate
coord, val = psf_prop.slices().x      # microns, normalized intensity
airy = psfmod.airydisk(np.abs(coord), FNO, WVL)
airy = np.nan_to_num(airy, nan=1.0)   # fill the r=0 hole with the known peak, 1

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(coord, val, label='propagated PSF')
ax.plot(coord, airy, 'k--', lw=1, label='analytic Airy')
ax.set(yscale='log', ylim=(1e-5, 2), xlim=(-airy_radius*6, airy_radius*6),
       xlabel='x [um]', ylabel='normalized intensity',
       title='propagated PSF vs analytic Airy disk')
ax.legend()

As you may expect, the agreement is excellent.  We can compare this PSF to
what you get when a central aperture is introduced:

## A central obscuration

Central obscurations can be implemented either by subtracting an inner circle
from an outer one, or calling the specialty `annulus` function.  We'll use the
latter here to model a 35% linear obscuration:

In [ ]:
eps = 0.35  # linear obscuration ratio
A_obsc = annulus(eps * EPD/2, EPD/2, r)

def psf_of(aperture, Q=4):
    wf = Wavefront(aperture.astype(complex), WVL, dx)
    p = wf.focus(EFL, Q=Q).intensity
    p.data = p.data / p.data.max()
    return p

p_clear = psf_of(A)
p_obsc = psf_of(A_obsc)

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(6, 6))
axs[0,0].imshow(A, cmap='gray')
axs[0,1].imshow(A_obsc, cmap='gray')
p_clear.plot2d(xlim=airy_radius*6, power=1/3, cmap='gray', fig=fig, ax=axs[1,0], show_colorbar=False)
axs[0,0].set_title('clear aperture')
p_obsc.plot2d(xlim=airy_radius*6, power=1/3, cmap='gray', fig=fig, ax=axs[1,1], show_colorbar=False)
axs[0,1].set_title(f'{eps:.0%} central obscuration')
fig.tight_layout()

In [ ]:
cc, vc = p_clear.slices().x
co, vo = p_obsc.slices().x
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cc, vc, label='clear')
ax.plot(co, vo, label='obscured')
ax.set(yscale='log', ylim=(1e-5, 2), xlim=(-airy_radius*6, airy_radius*6),
       xlabel='x [um]', ylabel='normalized intensity',
       title='central obscuration pumps energy into the rings')
ax.legend();

## Spider struts

Spiders are available.  They are only modeled as thin rectangles.  For a more
complex spider, write your own function that operates on the grid.

In [ ]:
vanes = spider(4, 0.25, xi, eta)  # 4 vanes, 0.25 mm wide
A_strut = A_obsc * ~vanes
p_strut = psf_of(A_strut)

fig, axs = plt.subplots(1, 2, figsize=(9, 4))
axs[0].imshow(A_strut, extent=[-EPD/2, EPD/2, -EPD/2, EPD/2], cmap='gray')
axs[0].set(title='obscured aperture + 4 spider vanes', xlabel='xi [mm]')
p_strut.plot2d(xlim=airy_radius*18, power=1/4, cmap='gray', fig=fig, ax=axs[1], show_colorbar=False)
axs[1].set_title('PSF: diffraction spikes')
fig.tight_layout()

## A polygonal aperture

Segmented and faceted mirrors are generally polygonal.  All N-sided regular
polygons are available through `regular_polygon`.  Regular means all sides have
the same length.  We'll draw a hexagon here:

In [ ]:
A_hex = regular_polygon(6, EPD/2, xi, eta)
p_hex = psf_of(A_hex)

fig, axs = plt.subplots(1, 2, figsize=(9, 4))
axs[0].imshow(A_hex, extent=[-EPD/2, EPD/2, -EPD/2, EPD/2], cmap='gray')
axs[0].set(title='hexagonal aperture', xlabel='xi [mm]')
p_hex.plot2d(xlim=airy_radius*8, power=1/4, cmap='gray', fig=fig, ax=axs[1], show_colorbar=False)
axs[1].set_title('PSF: six-fold spikes')
fig.tight_layout()

## Anti-aliasing

Some applications are very sensitive to the quality of the mask drawn.  Thus
far, we have shown only binary masks.  prysm has a built-in `antialias`
function, which takes most of the geometries and draws an anti-aliased
(grayscale edge) version of it.  The user needs to use the `_sdf` or Signed
Distance Function helper for the given shape as an input.  This approach is
very fast, and is the generalization of Fienup's true circle shader, which
shipped in previous versions of prysm, to a wide class of shapes including
all regular polygons.

We'll demonstrate it on the annular aperture, at extremely
coarse sampling to emphasize the anti-aliasing quality:

In [ ]:
xi, eta = make_xy_grid(128, diameter=EPD)
r, t = cart_to_polar(xi, eta)

A_obsc = annulus(eps * EPD/2, EPD/2, r)
A_obsc_aa = antialias(annulus_sdf(eps * EPD/2, EPD/2, r), dx)
fig, axs = plt.subplots(ncols=2, figsize=(6,6))
axs[0].imshow(A_obsc);
axs[0].set_title('Binary')
axs[1].imshow(A_obsc_aa);
axs[1].set_title('Antialiased');

A second variant of anti-aliasing is also available via multisample AA.  It is
more suited to extremely low resolution outputs, while the _sdf is best suited
to refining an already fairly well sampled representation.  You can see that
at the 128 px resolution, MSAA does slightly better.  It should generally
be preferred if you are not sensitive to the runtime of drawing apertures.
Though, it is still quite fast unless you are doing MSAA at extremely large
grids on a computer with slow single-core performance.

In [ ]:
def aper(x, y):
    r = np.hypot(x, y)
    return annulus(eps * EPD/2, EPD/2, r)

A_obsc_msaa = multisample(aper, xi, eta, 8)
fig, axs = plt.subplots(ncols=2, figsize=(6,6))
axs[0].imshow(A_obsc);
axs[0].set_title('Binary')
axs[1].imshow(A_obsc_msaa);
axs[1].set_title('Antialiased');

## Wrap-up

This notebook showed a few different aperture generating functions, including
annular and polygonal apertures, as well as hexagonal apertures to show regular
polygons.  At the end, we touched on the two anti-aliasing strategies available
to the user, which become important in very sensitive applications such as
coronagraphy.

To this point we have always used FFTs for propagation.  Most models are
actually better served by using a discrete fourier transform, which is the
subject of the next notebook (103).